# Retail Data ELT Lab — SQLite and Python

**Student starter notebook**

Use this notebook with the lab instructions to load the instructor-provided CSV files into SQLite, inspect their schemas, create staging views, run data-quality checks, and answer business questions.

**Tools:** Python, pandas, SQLite, Google Colab. No BigQuery or dbt setup is required.

Before starting, put `customers.csv`, `products.csv`, `orders_raw.csv`, and `orders_clean.csv` in a folder in your Google Drive, or use the direct-upload option in Step 1. Run cells from top to bottom. Use actual column names from your files; examples below are templates.

In [4]:
import sqlite3
from pathlib import Path
import pandas as pd

print("Python and SQLite are ready.")
print("SQLite version:", sqlite3.sqlite_version)
print("pandas version:", pd.__version__)

Python and SQLite are ready.
SQLite version: 3.45.1
pandas version: 2.2.3


## 1. Make the CSV files available

Choose **one** option: Google Drive (recommended) or direct upload. If using Drive, copy the four CSV files into a folder you own, for example `MyDrive/Retail_ELT_Lab`.

In [ ]:
# OPTION A (recommended): Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/Retail_ELT_Lab')  # Change if your folder differs

required_files = [
    'customers.csv',
    'products.csv',
    'orders_raw.csv',
    'orders_clean.csv'
]

for filename in required_files:
    file_path = DATA_DIR / filename
    print(f"{filename}: {'FOUND' if file_path.exists() else 'NOT FOUND'}")

In [8]:
# OPTION B: Upload files directly from your computer.
# Run this cell ONLY if you are not using Google Drive.
from google.colab import files
uploaded = files.upload()
DATA_DIR = Path('/content')
#
# After upload, verify that all four CSV files are in /content.

Saving products.csv to products.csv


## 2. Read and inspect the CSV files

Run this after all four files show as found (or after direct upload). Review the columns and sample records before writing SQL.

In [9]:
customers_df = pd.read_csv(DATA_DIR / 'customers.csv')
products_df = pd.read_csv(DATA_DIR / 'products.csv')
orders_raw_df = pd.read_csv(DATA_DIR / 'orders_raw.csv')
orders_clean_df = pd.read_csv(DATA_DIR / 'orders_clean.csv')

dataframes = {
    'customers': customers_df,
    'products': products_df,
    'orders_raw': orders_raw_df,
    'orders_clean': orders_clean_df
}

for name, df in dataframes.items():
    print(f"\n--- {name} ---")
    print("Shape (rows, columns):", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head())


--- customers ---
Shape (rows, columns): (500, 7)
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'city', 'state', 'signup_date']


,customer_id,first_name,last_name,email,city,state,signup_date
0,C0001,James,Gupta,james.gupta1@example.com,Dallas,TX,2023-07-18
1,C0002,Krishna,Khan,krishna.khan2@example.com,Houston,TX,2022-07-29
2,C0003,Maria,Davis,maria.davis3@example.com,Austin,TX,2024-05-14
3,C0004,Aisha,Sharma,aisha.sharma4@example.com,Austin,TX,2023-03-24
4,C0005,Krishna,Brown,krishna.brown5@example.com,Boston,MA,2022-02-24



--- products ---
Shape (rows, columns): (20, 4)
Columns: ['product_id', 'product_name', 'category', 'price']


,product_id,product_name,category,price
0,P001,Wireless Headphones,Electronics,89.99
1,P002,Bluetooth Speaker,Electronics,49.99
2,P003,Laptop Stand,Electronics,34.99
3,P004,USB-C Hub,Electronics,29.99
4,P005,Smart Watch,Electronics,199.99



--- orders_raw ---
Shape (rows, columns): (5100, 6)
Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']


,order_id,customer_id,product_id,quantity,order_date,status
0,O02814,C0472,P011,4,2025-07-15,pending
1,O04017,C0492,P020,5,2025-04-30,pending
2,O03993,C0065,P001,5,2025-01-10,delivered
3,O04314,C0107,P011,1,2025-07-03,delivered
4,O01732,C0175,P019,2,2024-04-28,shipped



--- orders_clean ---
Shape (rows, columns): (5000, 6)
Columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']


,order_id,customer_id,product_id,quantity,order_date,status
0,O00001,C0397,P009,4,2025-09-18,delivered
1,O00002,C0147,P016,2,2024-03-16,delivered
2,O00003,C0232,P006,4,2025-11-29,cancelled
3,O00004,C0496,P011,3,2025-07-17,shipped
4,O00005,C0278,P010,3,2025-12-29,pending


In [10]:
for name, df in dataframes.items():
    print(f"\n--- {name}: data types ---")
    display(df.dtypes.to_frame('data_type'))
    print("Missing values by column:")
    display(df.isna().sum().to_frame('missing_count'))


--- customers: data types ---


,data_type
customer_id,object
first_name,object
last_name,object
email,object
city,object
state,object
signup_date,object


Missing values by column:


,missing_count
customer_id,0
first_name,0
last_name,0
email,0
city,0
state,0
signup_date,0



--- products: data types ---


,data_type
product_id,object
product_name,object
category,object
price,float64


... (6 outputs left out)


--- orders_clean: data types ---


,data_type
order_id,object
customer_id,object
product_id,object
quantity,int64
order_date,object
status,object


Missing values by column:


,missing_count
order_id,0
customer_id,0
product_id,0
quantity,0
order_date,0
status,0


## 3. Create a SQLite database and load the raw tables

Load `customers.csv`, `products.csv`, and `orders_raw.csv` into SQLite. Keep `orders_clean.csv` as a reference file for later comparison; do not use it to replace the raw orders table.

In [11]:
DB_PATH = Path('/content/retail_lab.db')
conn = sqlite3.connect(DB_PATH)

raw_dataframes = {
    'customers': customers_df,
    'products': products_df,
    'orders_raw': orders_raw_df
}

for table_name, df in raw_dataframes.items():
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Loaded {len(df):,} rows into {table_name}")

print("SQLite database:", DB_PATH)

Loaded 500 rows into customers
Loaded 20 rows into products
Loaded 5,100 rows into orders_raw
SQLite database: /content/retail_lab.db


In [12]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)
display(tables)

for table_name in ['customers', 'products', 'orders_raw']:
    count = pd.read_sql_query(
        f'SELECT COUNT(*) AS row_count FROM "{table_name}"',
        conn
    ).iloc[0]['row_count']
    print(f"{table_name}: {count:,} rows")

,name
0,customers
1,orders_raw
2,products


customers: 500 rows
products: 20 rows
orders_raw: 5,100 rows


In [13]:
display(pd.read_sql_query('SELECT * FROM orders_raw LIMIT 10;', conn))

,order_id,customer_id,product_id,quantity,order_date,status
0,O02814,C0472,P011,4,2025-07-15,pending
1,O04017,C0492,P020,5,2025-04-30,pending
2,O03993,C0065,P001,5,2025-01-10,delivered
3,O04314,C0107,P011,1,2025-07-03,delivered
4,O01732,C0175,P019,2,2024-04-28,shipped
5,O03546,C0343,P007,5,2024-01-27,delivered
6,O02403,C0189,P020,4,2025-07-28,pending
7,O04813,C0455,P015,3,2024-11-04,shipped
8,O01179,C0424,P010,3,2025-12-05,delivered
9,O00202,C0103,P019,5,2025-07-17,delivered


## 4. Inspect SQLite schemas and table grain

Review the exact field names and data types. Decide what one row represents in each table (for example, one customer, one product, one order, or one order line).

In [14]:
for table_name in ['customers', 'products', 'orders_raw']:
    print(f"\n--- {table_name} schema ---")
    display(pd.read_sql_query(f'PRAGMA table_info("{table_name}");', conn))


--- customers schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,TEXT,0,None,0
1,1,first_name,TEXT,0,None,0
2,2,last_name,TEXT,0,None,0
3,3,email,TEXT,0,None,0
4,4,city,TEXT,0,None,0
5,5,state,TEXT,0,None,0
6,6,signup_date,TEXT,0,None,0



--- products schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,product_id,TEXT,0,None,0
1,1,product_name,TEXT,0,None,0
2,2,category,TEXT,0,None,0
3,3,price,REAL,0,None,0



--- orders_raw schema ---


,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,customer_id,TEXT,0,None,0
2,2,product_id,TEXT,0,None,0
3,3,quantity,INTEGER,0,None,0
4,4,order_date,TEXT,0,None,0
5,5,status,TEXT,0,None,0


### Table Grain and Candidate Keys

- **Customers:** Each row represents one customer. Candidate key: `customer_id`.
- **Products:** Each row represents one product. Candidate key: `product_id`.
- **Orders:** Each row represents one order record containing a customer, product, quantity, date, and status. Candidate key: `order_id` is not unique in the raw dataset because duplicate records exist.

The raw orders data contains exact duplicate records, so duplicate checks are necessary before analysis.

## 5. Create staging views with SQL

These starter views select all columns so they can run before you know the final field names. Improve them by selecting fields needed for analysis, standardizing text where appropriate, and documenting quality rules. If you edit a view definition, rerun its cell; each cell drops and recreates the view.

In [15]:
conn.execute("DROP VIEW IF EXISTS stg_customers;")

conn.execute("""
CREATE VIEW stg_customers AS
SELECT
    TRIM(customer_id) AS customer_id,
    TRIM(first_name) AS first_name,
    TRIM(last_name) AS last_name,
    LOWER(TRIM(email)) AS email,
    TRIM(city) AS city,
    UPPER(TRIM(state)) AS state,
    NULLIF(TRIM(signup_date), '') AS signup_date
FROM customers;
""")

conn.commit()

display(pd.read_sql_query(
    'SELECT * FROM stg_customers LIMIT 5;', conn
))

,customer_id,first_name,last_name,email,city,state,signup_date
0,C0001,James,Gupta,james.gupta1@example.com,Dallas,TX,2023-07-18
1,C0002,Krishna,Khan,krishna.khan2@example.com,Houston,TX,2022-07-29
2,C0003,Maria,Davis,maria.davis3@example.com,Austin,TX,2024-05-14
3,C0004,Aisha,Sharma,aisha.sharma4@example.com,Austin,TX,2023-03-24
4,C0005,Krishna,Brown,krishna.brown5@example.com,Boston,MA,2022-02-24


In [16]:
conn.execute("DROP VIEW IF EXISTS stg_products;")

conn.execute("""
CREATE VIEW stg_products AS
SELECT
    TRIM(product_id) AS product_id,
    TRIM(product_name) AS product_name,
    TRIM(category) AS category,
    CAST(price AS REAL) AS price
FROM products;
""")

conn.commit()

display(pd.read_sql_query(
    'SELECT * FROM stg_products LIMIT 5;', conn
))

,product_id,product_name,category,price
0,P001,Wireless Headphones,Electronics,89.99
1,P002,Bluetooth Speaker,Electronics,49.99
2,P003,Laptop Stand,Electronics,34.99
3,P004,USB-C Hub,Electronics,29.99
4,P005,Smart Watch,Electronics,199.99


In [17]:
conn.execute("DROP VIEW IF EXISTS stg_orders;")

conn.execute("""
CREATE VIEW stg_orders AS
SELECT
    TRIM(order_id) AS order_id,
    NULLIF(TRIM(customer_id), '') AS customer_id,
    TRIM(product_id) AS product_id,
    CAST(quantity AS INTEGER) AS quantity,
    NULLIF(TRIM(order_date), '') AS order_date,
    CASE
        WHEN UPPER(TRIM(status)) = 'IN_TRANSIT!!'
        THEN 'in_transit'
        ELSE LOWER(TRIM(status))
    END AS status
FROM orders_raw;
""")

conn.commit()

display(pd.read_sql_query(
    'SELECT * FROM stg_orders LIMIT 10;', conn
))

,order_id,customer_id,product_id,quantity,order_date,status
0,O02814,C0472,P011,4,2025-07-15,pending
1,O04017,C0492,P020,5,2025-04-30,pending
2,O03993,C0065,P001,5,2025-01-10,delivered
3,O04314,C0107,P011,1,2025-07-03,delivered
4,O01732,C0175,P019,2,2024-04-28,shipped
5,O03546,C0343,P007,5,2024-01-27,delivered
6,O02403,C0189,P020,4,2025-07-28,pending
7,O04813,C0455,P015,3,2024-11-04,shipped
8,O01179,C0424,P010,3,2025-12-05,delivered
9,O00202,C0103,P019,5,2025-07-17,delivered


In [18]:
display(pd.read_sql_query("""
    SELECT DISTINCT status
    FROM stg_orders
    ORDER BY status;
""", conn))

,status
0,cancelled
1,delivered
2,in_transit
3,pending
4,shipped


### Example transformation pattern — adapt to your schema

This is only an example; it will not run unless the fields exist in your table.

```sql
SELECT
    order_id,
    product_id,
    quantity,
    unit_price,
    CAST(quantity AS REAL) * CAST(unit_price AS REAL) AS line_total
FROM orders_raw;
```

SQLite casts can behave unexpectedly for invalid text values. Inspect the raw values and document conversion rules before relying on calculated values.

## 6. Data-quality checks

The examples below assume `customer_id` exists and that the customers table should have one row per customer. Change the field name or rule to match your actual schema.

In [ ]:
# Example: missing or blank customer IDs.
# Uncomment and edit customer_id if that is the actual identifier column.
#
# display(pd.read_sql_query("""
# SELECT COUNT(*) AS missing_customer_ids
# FROM stg_customers
# WHERE customer_id IS NULL
#    OR TRIM(CAST(customer_id AS TEXT)) = '';
# """, conn))

In [ ]:
# Example: duplicate customer IDs (only if one row per customer is expected).
#
# display(pd.read_sql_query("""
# SELECT customer_id, COUNT(*) AS records
# FROM stg_customers
# GROUP BY customer_id
# HAVING COUNT(*) > 1;
# """, conn))

In [19]:
display(pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_orders,
        SUM(CASE
            WHEN customer_id IS NULL OR TRIM(customer_id) = ''
            THEN 1 ELSE 0
        END) AS missing_customer_ids
    FROM stg_orders;
""", conn))

,total_orders,missing_customer_ids
0,5100,173


Out of 5,100 order records, 173 have missing customer IDs. These records cannot be directly linked to a customer, so they require further investigation. I have retained them in the staging view instead of deleting them.

In [20]:
display(pd.read_sql_query("""
    SELECT
        order_id,
        COUNT(*) AS occurrences
    FROM stg_orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
    LIMIT 10;
""", conn))

,order_id,occurrences
0,O04994,2
1,O04974,2
2,O04949,2
3,O04864,2
4,O04771,2
5,O04770,2
6,O04691,2
7,O04673,2
8,O04666,2
9,O04618,2


Some order IDs appear more than once in the raw orders data. This may be expected if an order contains multiple product records. Therefore, duplicate order IDs should not automatically be treated as errors. Further investigation is needed to understand the table's grain.

In [21]:
display(pd.read_sql_query("""
    SELECT *
    FROM stg_orders
    WHERE order_id = 'O04994';
""", conn))

,order_id,customer_id,product_id,quantity,order_date,status
0,O04994,C0242,P004,5,2024-07-06,pending
1,O04994,C0242,P004,5,2024-07-06,pending


The order ID O04994 appears twice with identical values across all columns. This indicates an exact duplicate record and may lead to overcounting in sales analysis. Such records should be investigated before calculating final business metrics.

In [22]:
display(pd.read_sql_query("""
    SELECT
        COUNT(*) AS duplicate_groups,
        SUM(occurrences - 1) AS extra_duplicate_rows
    FROM (
        SELECT COUNT(*) AS occurrences
        FROM stg_orders
        GROUP BY order_id, customer_id, product_id,
                 quantity, order_date, status
        HAVING COUNT(*) > 1
    );
""", conn))

,duplicate_groups,extra_duplicate_rows
0,100,100


terpretation to add to your notebook:
The analysis identified 100 groups of exact duplicate order records, resulting in 100 extra rows. These duplicates may cause inaccurate order counts and sales calculations. They should be investigated and handled before final analysis.

## 7. Analytical SQL — answer business questions

Use your actual column names. First inspect `stg_orders`, `stg_customers`, and `stg_products` and identify:
- the order identifier and customer identifier
- the product identifier (if each row represents an order line)
- the order date
- the quantity, unit price, discount, and line total (where available)
- any order status, region, or category fields

**Important:** The queries below are templates, not guaranteed to run unchanged. Replace placeholder column names with fields in your files. If a field is not available, explain that limitation and choose a suitable alternative. Run at least four queries and write 1–3 sentences interpreting each result.


In [ ]:
# Query A — overall sales and order-line summary
# Adapt the placeholders to your schema. If line_total is absent, calculate it
# from quantity * unit_price (and apply discounts only if their meaning is clear).
#
# display(pd.read_sql_query("""
# SELECT
#     COUNT(*) AS order_line_count,
#     COUNT(DISTINCT order_id) AS distinct_orders,
#     SUM(quantity) AS units_sold,
#     ROUND(SUM(line_total), 2) AS total_sales,
#     ROUND(AVG(line_total), 2) AS average_line_value
# FROM stg_orders;
# """, conn))
#
# Interpretation prompts:
# 1. How many rows are present compared with distinct orders?
# 2. Why might these counts be different?
# 3. What does the average line value tell you (and what does it not tell you)?


In [ ]:
# Query B — product performance (join orders to products)
# Confirm that the join keys match and that the product table has one row per product.
#
# display(pd.read_sql_query("""
# SELECT
#     p.product_name,
#     SUM(o.quantity) AS units_sold,
#     ROUND(SUM(o.line_total), 2) AS product_sales,
#     COUNT(DISTINCT o.order_id) AS orders_including_product
# FROM stg_orders AS o
# JOIN stg_products AS p
#     ON o.product_id = p.product_id
# GROUP BY p.product_name
# ORDER BY product_sales DESC
# LIMIT 10;
# """, conn))
#
# Query C — customer-level performance
# Adapt names and decide whether customers with no orders should be included.
#
# display(pd.read_sql_query("""
# SELECT
#     c.customer_id,
#     COUNT(DISTINCT o.order_id) AS order_count,
#     ROUND(SUM(o.line_total), 2) AS customer_sales,
#     ROUND(AVG(o.line_total), 2) AS average_line_value
# FROM stg_customers AS c
# LEFT JOIN stg_orders AS o
#     ON c.customer_id = o.customer_id
# GROUP BY c.customer_id
# ORDER BY customer_sales DESC;
# """, conn))
#
# Query D — monthly sales trend (requires a date column in YYYY-MM-DD format)
#
# display(pd.read_sql_query("""
# SELECT
#     substr(order_date, 1, 7) AS order_month,
#     COUNT(DISTINCT order_id) AS order_count,
#     ROUND(SUM(line_total), 2) AS monthly_sales
# FROM stg_orders
# WHERE order_date IS NOT NULL
# GROUP BY substr(order_date, 1, 7)
# ORDER BY order_month;
# """, conn))
#
# Query E — investigate order identifiers that appear more than once
# (This can be normal for line-item data; inspect the table grain before deciding.)
#
# display(pd.read_sql_query("""
# SELECT order_id, COUNT(*) AS rows_per_order
# FROM stg_orders
# GROUP BY order_id
# HAVING COUNT(*) > 1
# ORDER BY rows_per_order DESC
# LIMIT 20;
# """, conn))


In [23]:
display(pd.read_sql_query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS distinct_orders,
        SUM(quantity) AS total_units_sold
    FROM stg_orders;
""", conn))

,total_rows,distinct_orders,total_units_sold
0,5100,5000,14947


The raw orders dataset contains 5,100 rows representing 5,000 distinct order IDs. The total quantity recorded is 14,947 units. The difference between total rows and distinct orders indicates that some order IDs occur more than once. Exact duplicate records identified earlier may affect these totals.

In [24]:
display(pd.read_sql_query("""
    SELECT
        SUM(o.quantity * p.price) AS total_sales,
        AVG(o.quantity * p.price) AS average_line_value
    FROM stg_orders o
    JOIN stg_products p
        ON o.product_id = p.product_id;
""", conn))

,total_sales,average_line_value
0,1113264.53,218.287163


The total calculated sales value is 1,113,264.53, and the average line value is approximately 218.29. Sales were calculated by multiplying the order quantity by the product price and joining the orders and products tables using product_id.

In [25]:
display(pd.read_sql_query("""
    SELECT
        p.product_name,
        p.category,
        SUM(o.quantity) AS total_units_sold,
        ROUND(SUM(o.quantity * p.price), 2) AS total_sales
    FROM stg_orders o
    JOIN stg_products p
        ON o.product_id = p.product_id
    GROUP BY p.product_id, p.product_name, p.category
    ORDER BY total_sales DESC
    LIMIT 10;
""", conn))

,product_name,category,total_units_sold,total_sales
0,Smart Watch,Electronics,926,185190.74
1,Cookware Set,Home,648,103673.52
2,Winter Jacket,Apparel,686,102893.14
3,Running Shoes,Apparel,746,96972.54
4,Mechanical Keyboard,Electronics,732,87832.68
5,Tennis Racket,Sports,804,80391.96
6,Wireless Headphones,Electronics,760,68392.40
7,Coffee Maker,Home,854,68311.46
8,Dumbbell Pair,Sports,739,51722.61
9,Denim Jeans,Apparel,748,44872.52


Smart Watch is the highest revenue-generating product, with total sales of 185,190.74 and 926 units sold. Cookware Set and Winter Jacket rank second and third in sales. This analysis helps identify the products contributing most to overall revenue.

In [26]:
display(pd.read_sql_query("""
    SELECT
        c.customer_id,
        c.first_name || ' ' || c.last_name AS customer_name,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(o.quantity * p.price), 2) AS total_spent
    FROM stg_orders o
    JOIN stg_customers c
        ON o.customer_id = c.customer_id
    JOIN stg_products p
        ON o.product_id = p.product_id
    GROUP BY c.customer_id, customer_name
    ORDER BY total_spent DESC
    LIMIT 10;
""", conn))

,customer_id,customer_name,total_orders,total_spent
0,C0152,Nikhil Garcia,13,6777.42
1,C0103,Liam Kim,13,5328.54
2,C0232,Aditya Miller,18,5269.37
3,C0301,Neha Johnson,17,5219.44
4,C0192,Nikhil Das,14,4899.47
5,C0403,Olivia Garcia,18,4879.38
6,C0104,Riya Mehta,14,4842.55
7,C0497,Sara Kim,15,4814.48
8,C0384,Ishaan Mehta,13,4794.62
9,C0198,Maria Miller,12,4779.56


The customer analysis identifies the top 10 customers based on total spending. Nikhil Garcia (C0152) is the highest-spending customer, with total purchases of 6,777.42 across 13 distinct orders. Aditya Miller (C0232) has the highest order count among the top 10, with 18 orders. This analysis helps identify valuable customers and understand purchasing behaviour.

In [27]:
display(pd.read_sql_query("""
    SELECT
        strftime('%Y-%m', order_date) AS sales_month,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(quantity) AS total_units_sold,
        ROUND(SUM(quantity * price), 2) AS total_sales
    FROM stg_orders
    JOIN stg_products USING (product_id)
    WHERE date(order_date) IS NOT NULL
    GROUP BY sales_month
    ORDER BY sales_month;
""", conn))

,sales_month,total_orders,total_units_sold,total_sales
0,2024-01,210,654,52877.46
1,2024-02,197,569,41020.31
2,2024-03,196,628,46057.72
3,2024-04,212,612,43472.88
4,2024-05,215,643,46684.57
5,2024-06,203,594,47876.06
6,2024-07,204,596,48946.04
7,2024-08,199,606,44149.94
8,2024-09,211,654,44756.46
9,2024-10,203,583,39988.17


The monthly sales analysis shows that sales varied throughout the two-year period. August 2025 recorded the highest monthly sales at 62,345.85, while October 2024 recorded the lowest at 39,988.17. July 2025 also showed strong performance, with 720 units sold. This analysis helps identify monthly sales patterns and periods of stronger business performance.

In [28]:
display(pd.read_sql_query("""
    SELECT
        order_id,
        COUNT(*) AS product_records,
        COUNT(DISTINCT product_id) AS distinct_products,
        SUM(quantity) AS total_units
    FROM stg_orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY product_records DESC
    LIMIT 10;
""", conn))

,order_id,product_records,distinct_products,total_units
0,O04994,2,1,10
1,O04974,2,1,8
2,O04949,2,1,8
3,O04864,2,1,8
4,O04771,2,1,2
5,O04770,2,1,2
6,O04691,2,1,10
7,O04673,2,1,10
8,O04666,2,1,10
9,O04618,2,1,8


The order-level investigation identified repeated order IDs with the same product appearing more than once. The inspected records are exact duplicates rather than separate product lines. These duplicates can inflate unit counts and sales totals, so they should be investigated and removed from the analytical dataset only after validation.

## 8. Reconcile your results with `orders_clean.csv`

Treat `orders_clean.csv` as a reference dataset. Do not assume it has the same grain as `orders_raw` or your staging view. First inspect the fields and determine whether each row represents an order or an order line.

Complete these checks:
1. Compare row counts and column names.
2. Check for missing and duplicate business keys in the reference file.
3. Compare totals only when the two datasets use the same grain and the same definition of sales.
4. If both datasets share a reliable key, compare key coverage and identify records present in only one dataset.
5. If you compare numeric values, report the absolute difference and use a sensible rounding tolerance.
6. Explain at least two differences or data-quality issues you found. If the files cannot be compared directly, explain why and define the transformation needed to make them comparable.


In [29]:
# Basic profile of the reference dataset
print("Reference row count:", len(orders_clean_df))
print("Reference columns:", orders_clean_df.columns.tolist())
print("\\nMissing values by column:")
display(orders_clean_df.isna().sum().rename("missing_count").to_frame())
print("\\nDuplicate full rows:", int(orders_clean_df.duplicated().sum()))
display(orders_clean_df.head())

# TODO 1: Identify the business key (for example, order_id or order_line_id)
# in the reference data and check whether it is missing or duplicated.
#
# Example template — replace order_id with the correct key:
# reference_key = "order_id"
# print("Missing reference keys:", int(orders_clean_df[reference_key].isna().sum()))
# print("Duplicate reference keys:", int(orders_clean_df[reference_key].duplicated().sum()))
#
# TODO 2: Create a comparable result from your staging data.
# Example template — adapt fields and the aggregation level:
#
# transformed_df = pd.read_sql_query("""
# SELECT order_id, ROUND(SUM(line_total), 2) AS calculated_sales
# FROM stg_orders
# GROUP BY order_id
# """, conn)
#
# TODO 3: If both datasets contain the same key, compare key coverage.
# Rename the reference key and value columns to match the transformed result first.
#
# comparison = transformed_df.merge(
#     orders_clean_df[[reference_key, "sales_amount"]].rename(
#         columns={reference_key: "order_id", "sales_amount": "reference_sales"}
#     ),
#     on="order_id",
#     how="outer",
#     indicator=True
# )
# print(comparison["_merge"].value_counts())
#
# TODO 4: For records matched in both datasets, calculate a numeric difference.
# matched = comparison[comparison["_merge"] == "both"].copy()
# matched["absolute_difference"] = (
#     matched["calculated_sales"] - matched["reference_sales"]
# ).abs()
# print("Rows differing by more than 0.01:",
#       int((matched["absolute_difference"] > 0.01).sum()))
# display(matched.sort_values("absolute_difference", ascending=False).head(10))
#
# Note: The templates remain commented because the CSV column names and grain
# must be confirmed before choosing keys or comparing totals.


Reference row count: 5000
Reference columns: ['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']
\nMissing values by column:


,missing_count
order_id,0
customer_id,0
product_id,0
quantity,0
order_date,0
status,0


\nDuplicate full rows: 0


,order_id,customer_id,product_id,quantity,order_date,status
0,O00001,C0397,P009,4,2025-09-18,delivered
1,O00002,C0147,P016,2,2024-03-16,delivered
2,O00003,C0232,P006,4,2025-11-29,cancelled
3,O00004,C0496,P011,3,2025-07-17,shipped
4,O00005,C0278,P010,3,2025-12-29,pending


In [30]:
print("Raw orders:", len(orders_raw_df))
print("Clean reference orders:", len(orders_clean_df))

print("\nRaw columns:")
print(list(orders_raw_df.columns))

print("\nClean reference columns:")
print(list(orders_clean_df.columns))

Raw orders: 5100
Clean reference orders: 5000

Raw columns:
['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']

Clean reference columns:
['order_id', 'customer_id', 'product_id', 'quantity', 'order_date', 'status']


Both datasets contain the same six columns, but the raw dataset has 5,100 rows while the clean reference contains 5,000 rows. This difference of 100 rows may be related to duplicate records or other cleaning operations. Further checks are needed to confirm the differences.

In [31]:
print("Raw duplicate order IDs:",
      orders_raw_df['order_id'].duplicated().sum())

print("Clean duplicate order IDs:",
      orders_clean_df['order_id'].duplicated().sum())

Raw duplicate order IDs: 100
Clean duplicate order IDs: 0


In [32]:
missing_comparison = pd.DataFrame({
    "raw_missing": orders_raw_df.isnull().sum(),
    "clean_missing": orders_clean_df.isnull().sum()
})

display(missing_comparison)

,raw_missing,clean_missing
order_id,0,0
customer_id,173,0
product_id,0,0
quantity,0,0
order_date,0,0
status,0,0


The raw dataset contains 173 missing customer IDs, while the clean reference dataset contains no missing values in this column. This suggests that the missing customer IDs were handled during the cleaning process. However, the exact method used to handle them is not established by this comparison alone.

In [33]:
raw_duplicates = orders_raw_df[
    orders_raw_df.duplicated(keep=False)
]

print("Rows involved in exact duplicates:",
      len(raw_duplicates))

print("Extra duplicate rows:",
      orders_raw_df.duplicated().sum())

print("Clean dataset exact duplicate rows:",
      orders_clean_df.duplicated().sum())

Rows involved in exact duplicates: 200
Extra duplicate rows: 100
Clean dataset exact duplicate rows: 0


The raw dataset contains 200 rows involved in exact duplicate groups, representing 100 extra duplicate rows. The clean reference dataset contains no exact duplicate rows. This confirms that duplicate removal accounts for the 100-row difference between the two datasets.

In [34]:
raw_deduplicated = orders_raw_df.drop_duplicates()

comparison_check = raw_deduplicated.merge(
    orders_clean_df,
    on="order_id",
    how="outer",
    suffixes=("_raw", "_clean"),
    indicator=True
)

print("Raw rows after deduplication:", len(raw_deduplicated))
print("Clean reference rows:", len(orders_clean_df))

print("Records only in raw:",
      (comparison_check["_merge"] == "left_only").sum())

print("Records only in clean:",
      (comparison_check["_merge"] == "right_only").sum())

Raw rows after deduplication: 5000
Clean reference rows: 5000
Records only in raw: 0
Records only in clean: 0


In [35]:
comparison = raw_deduplicated.merge(
    orders_clean_df,
    on="order_id",
    how="outer",
    suffixes=("_raw", "_clean"),
    indicator=True
)

print("Records present only in raw:",
      (comparison["_merge"] == "left_only").sum())

print("Records present only in clean:",
      (comparison["_merge"] == "right_only").sum())

matched = comparison[comparison["_merge"] == "both"]

for col in ["customer_id", "product_id", "quantity",
            "order_date", "status"]:
    different = (
        matched[f"{col}_raw"].fillna("<NULL>") !=
        matched[f"{col}_clean"].fillna("<NULL>")
    ).sum()

    print(f"{col} differences:", different)

Records present only in raw: 0
Records present only in clean: 0
customer_id differences: 169
product_id differences: 0
quantity differences: 51
order_date differences: 57
status differences: 97


Both datasets contain the same 5,000 order IDs, with no records missing from either dataset. However, differences were found in customer IDs (169), quantities (51), order dates (57), and statuses (97). Product IDs remained unchanged. These differences indicate that the clean reference dataset includes additional transformations beyond duplicate removal. The exact reasons for these changes require further investigation.

In [36]:
changed_records = comparison[
    (comparison["_merge"] == "both") &
    (
        (comparison["customer_id_raw"].fillna("<NULL>") !=
         comparison["customer_id_clean"].fillna("<NULL>")) |
        (comparison["quantity_raw"] != comparison["quantity_clean"]) |
        (comparison["order_date_raw"] != comparison["order_date_clean"]) |
        (comparison["status_raw"] != comparison["status_clean"])
    )
]

display(changed_records[[
    "order_id",
    "customer_id_raw", "customer_id_clean",
    "quantity_raw", "quantity_clean",
    "order_date_raw", "order_date_clean",
    "status_raw", "status_clean"
]].head(10))

,order_id,customer_id_raw,customer_id_clean,quantity_raw,quantity_clean,order_date_raw,order_date_clean,status_raw,status_clean
10,O00011,NaN,C0422,1,1,2024-11-20,2024-11-20,delivered,delivered
16,O00017,C0459,C0459,1,1,2025-10-12,2025-10-12,IN_TRANSIT!!,delivered
24,O00025,NaN,C0280,3,3,2025-12-02,2025-12-02,delivered,delivered
27,O00028,C0175,C0175,2,2,2024-11-22,2024-11-22,IN_TRANSIT!!,delivered
28,O00029,C0376,C0376,4,4,2025-01-14,2025-01-14,IN_TRANSIT!!,delivered
33,O00034,NaN,C0098,4,4,2025-04-23,2025-04-23,cancelled,cancelled
60,O00061,C0340,C0340,-5,5,2024-06-26,2024-06-26,delivered,delivered
72,O00073,NaN,C0123,1,1,2025-06-27,2025-06-27,delivered,delivered
77,O00078,NaN,C0031,5,5,2024-02-08,2024-02-08,delivered,delivered
83,O00084,NaN,C0436,4,4,2024-04-25,2024-04-25,delivered,delivered


The comparison identified changes in customer IDs, quantities, order dates, and order statuses. Some missing customer IDs were filled, negative quantities were converted to positive values, and invalid dates were corrected. The reference dataset also changed some statuses, such as IN_TRANSIT!! to delivered. My staging view standardizes this status to in_transit instead. This shows that the reference dataset and my staging transformations do not use exactly the same cleaning rules.

In [37]:
print("Raw missing customer IDs:",
      orders_raw_df["customer_id"].isna().sum())

print("Clean missing customer IDs:",
      orders_clean_df["customer_id"].isna().sum())

print("\nClean missing customer ID examples:")

display(
    orders_clean_df[
        orders_clean_df["customer_id"].isna()
    ].head(10)
)

Raw missing customer IDs: 173
Clean missing customer IDs: 0

Clean missing customer ID examples:


,order_id,customer_id,product_id,quantity,order_date,status


The raw dataset contains 173 missing customer IDs, whereas the clean reference dataset contains zero missing customer IDs. This indicates that missing customer identifiers were handled during the cleaning process.

In [38]:
print("Raw non-positive quantities:",
      (orders_raw_df["quantity"] <= 0).sum())

print("Clean non-positive quantities:",
      (orders_clean_df["quantity"] <= 0).sum())

Raw non-positive quantities: 53
Clean non-positive quantities: 0


The raw dataset contains 53 records with non-positive quantities, while the clean reference dataset contains none. This indicates that invalid quantity values were corrected or removed during the cleaning process. The earlier examples show that a quantity of -5 was converted to 5.

In [39]:
print("Raw invalid dates:",
      pd.to_datetime(
          orders_raw_df["order_date"],
          errors="coerce"
      ).isna().sum())

print("Clean invalid dates:",
      pd.to_datetime(
          orders_clean_df["order_date"],
          errors="coerce"
      ).isna().sum())

Raw invalid dates: 58
Clean invalid dates: 0


The raw dataset contains 58 invalid date values, whereas the clean reference dataset contains zero invalid dates. This indicates that invalid or unparseable dates were handled during the cleaning process.

The reconciliation between the raw and clean reference datasets identified several data-quality improvements. The clean dataset contains 5,000 rows compared with 5,100 in the raw dataset. It has no exact duplicate rows, missing customer IDs, non-positive quantities, or invalid dates. Further comparisons identified changes in customer IDs, quantities, dates, and order statuses, while product IDs remained unchanged. These findings suggest that the reference dataset underwent multiple cleaning operations before being prepared for analysis.

# Final Findings Summary — Retail Data ELT Lab

## 1. Data Quality Findings

The raw retail dataset contained 5,100 order records. During data-quality checks, I identified:

- 173 missing customer IDs.
- 100 extra exact duplicate records.
- 53 non-positive quantities.
- 58 invalid order dates.

The clean reference dataset contains 5,000 records and has no missing customer IDs, exact duplicate records, non-positive quantities, or invalid dates.

## 2. Analytical Findings

- Total recorded units sold: 14,947.
- Calculated total sales: 1,113,264.53.
- Highest revenue-generating product: Smart Watch, with sales of 185,190.74.
- Highest-spending customer: Nikhil Garcia, with purchases worth 6,777.42.
- Highest monthly sales: August 2025, with 62,345.85.

## 3. Reconciliation Findings

Both datasets contain the same six columns and 5,000 matching order IDs after duplicate removal. However, differences were identified in customer IDs, quantities, order dates, and order statuses. Product IDs remained unchanged.

These findings indicate that the clean reference dataset underwent several data-cleaning and transformation operations.

The reference dataset also contains changes in customer IDs (169), quantities (51), order dates (57), and order statuses (97). These differences indicate that additional cleaning rules were applied beyond duplicate removal. Product IDs remained unchanged.

## 4. Conclusion

Through this lab, I practised loading retail CSV files into SQLite, creating SQL staging views, performing data-quality checks, writing analytical SQL queries, and reconciling raw data with a clean reference dataset. This exercise helped me understand the practical ELT workflow and the importance of data quality before performing business analysis.

## 9. Save your work

Save the notebook to Drive using **File → Save a copy in Drive**. Download the SQLite database before the Colab runtime ends.

In [ ]:
conn.commit()
# Run this when you have finished using the database connection.
# conn.close()

from google.colab import files
files.download('/content/retail_lab.db')

## Student checklist

- [ ] Inspected all four CSV files.
- [ ] Loaded three raw tables into SQLite.
- [ ] Documented actual schemas and table grain.
- [ ] Created and improved all three staging views using actual field names.
- [ ] Ran data-quality checks and explained decisions.
- [ ] Answered at least three business questions with SQL.
- [ ] Compared results with `orders_clean.csv`.
- [ ] Saved the notebook and downloaded `retail_lab.db`.